In [ ]:
##TODO
#Preprocessing of Data
#1. Remove ChatGPTisms - DONE
#2. Remove empty pages - DONE
#3. Split into sentences
#4. Remove short sentences? 
#5. Remove City Names, etc

In [2]:
import json
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

import spacy
from spacy.language import Language
from spacy.tokens import Span

# Load the spacy model
nlp = spacy.load('en_core_web_sm')

/var/folders/h1/_zymb6k15855dxzmpdj7099w0000gp/T/ipykernel_1265/4239076210.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/Users/modekurt/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [8]:
##Get Text 
with open('corpus/all_plans_text.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
        file.close()

with open('corpus/all_plans_translated_text.json', 'r', encoding='utf-8') as file:
       t_data = json.load(file)
       file.close()

#combine into one file
all_text = {}
for doc in data["english"]:
        all_text[doc] = data["english"][doc]["original_text"]
all_text = all_text | t_data

# Store translated text
with open('corpus/all_text.json', 'w', encoding='utf-8') as file:
        json.dump(all_text, file)
        file.close()      

In [ ]:
#Implementing tf-idf
text_json = 'corpus/all_text.json'

#Get all text
with open(text_json, 'r') as file:
    data = json.load(file)
corpus = [page for doc in data for page in data[doc]]


vectorizer = TfidfVectorizer()
corpus_tfidf = vectorizer.fit_transform(corpus)

# Display the TF-IDF matrix
df_tfidf = pd.DataFrame(corpus_tfidf.toarray(), columns=vectorizer.get_feature_names_out())

# Sum the TF-IDF scores for each word across all documents
word_sums = df_tfidf.sum(axis=0)

# Sort the words by their total TF-IDF scores
sorted_word_sums = word_sums.sort_values(ascending=False)

# Convert to DataFrame for easier viewing
df_sorted_words = pd.DataFrame(sorted_word_sums, columns=['TF-IDF Score'])

# Print the sorted words (optional)
print(df_sorted_words.head(10))

# Save the sorted words to a CSV file
df_sorted_words.to_csv('corpus/tfidf_all_text.csv', index=True)

In [3]:
##Remove ChatGPT sentences
chatgpt_sent = ["Bonjour, comment puis-je vous aider aujourd'hui?", "Bonjour, comment ça va aujourd'hui?"]
##TODO: Add: "Hello! How are you doing today?\", 
# ""The text provided seems to be a mix of random characters and symbols, making it difficult to translate accurately. If you have a specific portion of the text that you would like translated, please provide that part so I can assist you better.""
# ""Hello world!\""
# ""Hello! \n\nThis is a test message. \n\nThank you!\""

with open('corpus/all_text.json', 'r', encoding='utf-8') as file:
        all_text = json.load(file)
        file.close()

for doc in all_text:
    print(doc)
    removed_pages = 0
    for page in all_text[doc]:
        #Remove ChatGPT sentences
        for sent in chatgpt_sent:
            if sent in page: 
                print(page)
                all_text[doc].remove(page)
                removed_pages += 1
        #Remove empty pages
        if page.strip() == '':
            print(page)
            all_text[doc].remove(page)
            removed_pages += 1
    print(f"removed pages:{removed_pages}")

20140-2017-Yerevan-GCAP.pdf
removed pages:0
206171-2023-PPS22-036-Climate-Strategy-Document_2022_A5P_16pages_v10.pdf
removed pages:0
206171-2016-adaptwest-regional-adaptation-plan-october-2016 (1).pdf
  
 

removed pages:1
206171-2013-CLIMATE CHANGE ADAPTATION PLAN 2013-2015.pdf
removed pages:0
206171-2016-resilient-hills-and-coasts-climate-change-adaption-plan-gen.pdf
removed pages:0
206171-2016-climate_change_adaptation_plan_for_northern_adelaide_region.pdf
removed pages:0
206171-2016-Regional+Adaptation+Plan.pdf
removed pages:0
206171-2014-resilient-south-climate-change-adaptation-plan.pdf
removed pages:0
206171-2021-action-plan-water-sensitive-city.pdf
removed pages:0
206171-2023-7aafb2a9789831f654e2cefd482b3e0e_PPS23-097_Integrated_Climate_Strategy__A4_WEB.pdf
removed pages:0
206170-2016-20161128-flood_action_plan-update-nov_2016.pdf
removed pages:0
206170-2023-20240202-Brisbane-Clean-Green-Sustainable.pdf
removed pages:0
206170-2007-Attach5brf180907.pdf
removed pages:0
206170-202

In [4]:
# List of abbreviations that should not be treated as sentence boundaries
abbreviations = ['Dr.', 'e.g.', 'Mr.', 'Mrs.', 'Ms.', 'Prof.', 'Inc.', 'vs.', 'etc.']

@Language.component('custom_sentencizer')
def custom_sentencizer(doc):
    for i, token in enumerate(doc[:-2]):
        # Check if token is a potential sentence boundary
        if token.text in abbreviations:
            doc[i + 1].is_sent_start = False
    return doc

# Add the custom sentence segmentation rule to the pipeline
nlp.add_pipe('custom_sentencizer', before='parser')

<function __main__.custom_sentencizer(doc)>

In [5]:
#Split into sentences

all_text_sent = {}
for plan in all_text:
    pages = []
    for page in all_text[plan]:
            # print(page)
        doc = nlp(page)
        sentences = [sent.text for sent in doc.sents]
            # sentences = [sentence for split_sentences in sentences for sentence in split_sentences] #flatten list of sentences
        sentences = [sentence.replace('\n', ' ') for sentence in sentences]
        pages += sentences
    all_text_sent[plan] = pages

# Store translated text
with open('corpus/all_text_sentences.json', 'w', encoding='utf-8') as file:
        json.dump(all_text_sent, file)
        file.close() 

In [8]:
##TODO: rerun after changing to sentences
#remove stop words
# Function to remove stopwords from a sentence
# Function to remove stopwords from a sentence
def remove_stopwords(doc):
    cleaned_tokens = [token.text for token in doc if not token.is_stop]
    return " ".join(cleaned_tokens)

# Prepare a list of all sentences
all_sentences = [(key, sentence) for key, sentences in all_text_sent.items() for sentence in sentences]

# Process all sentences using nlp.pipe for efficient batch processing
docs = list(nlp.pipe([sentence for _, sentence in all_sentences]))

# Organize the results back into the original structure
cleaned_text = {}
for (key, _), doc in zip(all_sentences, docs):
    cleaned_sentence = remove_stopwords(doc)
    if key not in cleaned_text:
        cleaned_text[key] = []
    cleaned_text[key].append(cleaned_sentence)
print(cleaned_text[0][:10])

KeyError: 0

In [9]:
#print cleaned text
print(cleaned_text["20140-2017-Yerevan-GCAP.pdf"][:10])

[' \n                         OFFICIAL USE \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n      \n \n \n DRAFT \n \n ( 3 July 2017 ) \n \n Yerevan Green City Action Plan \n \n Yerevan 2017 \n', 'Yerevan Green City Action Plan \n 1 \n OFFICIAL USE \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Disclaimer \n Green City Action Plan prepared City Yerevan international team experts \n led Ernst & Young , s.r.o . ( Czech Republic ) . members consortium included GEOtest , \n SWECO , SEVEn local experts .  \n European Bank Reconstruction Development ( EBRD ) , Czech Government Official \n Development Assistance Technical Cooperation Fund City Yerevan carry \n responsibility selection , involvement monitoring Ernst & Young / party \n claims EBRD utilizing services provided Ernst & Young . \n', '2 \n OFFICIAL USE \n Executive Summary \n light continuous global urbanization , sustainable development challenges